# 34 — Mask-guided inpainting baseline (revision round 1, reviewer R2-4)

Same 298 crops and masks as MRT-GAN, 512 px. Train crops only; the test split is not read.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/inpaint_baseline_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# Study (p100 crops and configs) from kernel 13 output; three G3 tuning configs written next to G0-G3.
if torch.cuda.device_count()<1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count())
source=unique('study.json')
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
P100=STUDY/'p100'
sys.path.insert(0,str(HERE))
from melon_cyclegan import texture_study as s
from melon_cyclegan import texture_study_eval as e
scope,m=s.verify_scope(STUDY,100,DATASET)
g0=json.loads((P100/'configs/G0.json').read_text()); g3=json.loads((P100/'configs/G3.json').read_text())
assert g0['subset_digest']==g3['subset_digest']==m['subset_digest']
# R512 pool: same crop windows, same reviewed-pool membership and same GrabCut masks (nearest-upsampled),
# images re-cropped from the original photographs at 512 px. Only resolution changes.
def build_pool512(src, dst, dataset, size=512, require_train=True):
    import csv, json, collections, hashlib
    from pathlib import Path
    import numpy as np, cv2
    from PIL import Image
    from melon_cyclegan import config as cfg
    from melon_cyclegan.crops import crop_with_reflection
    from melon_cyclegan.color import phi_numpy
    from melon_cyclegan.training_data import digest, verify, sha
    src, dst, dataset = Path(src), Path(dst), Path(dataset)
    if dst.exists(): raise ValueError(f'{dst} exists')
    man = json.loads((src/'crops_manifest.json').read_text())
    rows = {r['id']: r for r in csv.DictReader(open(dataset/'manifest.csv'))}
    k = size/float(man.get('size', 256)); records=[]; feats=collections.defaultdict(list); drift=[]
    for r in man['records']:
        row = rows[r['id']]
        if (require_train and row['split']!='train') or sha(dataset/row['image'])!=r['source_sha256']: raise ValueError(f"{r['id']}: source photo differs")
        with Image.open(dataset/row['image']) as h: img=h.convert('RGB')
        left, top, side = r['crop_window']
        crop = cv2.resize(crop_with_reflection(img,left,top,side),(size,size),interpolation=cv2.INTER_AREA)
        m256 = Image.open(src/r['mask']).convert('L')
        mask = np.asarray(m256.resize((size,size),Image.Resampling.NEAREST))
        old = np.asarray(Image.open(src/r['image']).convert('RGB'),np.float64)
        drift.append(np.abs(cv2.resize(crop,old.shape[1::-1],interpolation=cv2.INTER_AREA).astype(np.float64)-old).mean())
        rec = dict(r, crop_box=[float(v)*k for v in r['crop_box']])
        for key, arr in (('image', crop), ('mask', mask)):
            p = dst/r[key]; p.parent.mkdir(parents=True, exist_ok=True); Image.fromarray(arr).save(p); rec[f'{key}_sha256']=sha(p)
        feats[int(r['class_id'])].append(phi_numpy(crop.astype(np.float64)/255.0, mask>127)); records.append(rec)
    if max(drift) > 4.0: raise ValueError(f'512 crops do not match the 256 pool geometry (max mean abs diff {max(drift):.2f})')
    classes = {}
    for cid, v in sorted(feats.items()):
        x=np.stack(v); classes[cfg.CLASS_NAMES[cid]]=dict(class_id=cid,count=len(x),centroid=x.mean(0).tolist(),spread=x.std(0).tolist())
    new = {k2: v for k2, v in man.items() if k2 not in ('records','crops_digest','centroids_sha256')}
    new.update(records=records, size=size, crops_digest=digest(records), source_crops_digest_256=man['crops_digest'],
               note='R512: same windows, members and GrabCut masks as the 256 pool; images re-cropped at 512')
    (dst/'color_centroids.json').write_text(json.dumps(dict(version=cfg.VERSION, phi=man['phi'],
        source='curated train split real crops (R512)', crops_digest=new['crops_digest'], classes=classes), indent=2)+'\n')
    new['centroids_sha256'] = sha(dst/'color_centroids.json')
    (dst/'crops_manifest.json').write_text(json.dumps(new, indent=2)+'\n')
    verify(dst, man['training_policy'])
    return new, dict(max_drift=float(max(drift)), mean_drift=float(np.mean(drift)), n=len(records))

POOL512=P100/'crops512'
pool_manifest,pool_check=build_pool512(P100/'crops',POOL512,DATASET)
print('R512 pool',pool_check,pool_manifest['crops_digest'])
sys.path.insert(0,str(HERE/'tools'))
from calibrate_color_margin import calibrate
report=calibrate(POOL512); (POOL512/'color_margin.json').write_text(json.dumps(report,indent=2)+'\n')
print('margin 512',report['margin'],'margin 256',json.loads((P100/'crops/color_margin.json').read_text())['margin'])
if report['margin']<=0: raise ValueError('nonpositive margin at 512')
from melon_cyclegan.dataset_impl import load_pair
from melon_cyclegan.texture import patch_indices
for r in pool_manifest['records']:
    _,mk=load_pair(POOL512/r['image'],POOL512/r['mask']); patch_indices(mk[None],64,32,4,.95,random=False)
g3ti=dict(g3,composite_mask=0,lambda_identity=0.5,load_size=512,crop_size=512,texture_size=64,texture_stride=32,
          expected_crops_digest=pool_manifest['crops_digest'])


In [ ]:
# Isolated Python 3.10 environment; official diffusers LoRA script of the same release as the library.
PY='/tmp/tenv/bin/python'; TAG='v0.30.3'; DF=Path('/tmp/diffusers')
sh=lambda cmd: subprocess.run(cmd,shell=True,check=True)
sh('pip install -q uv && uv venv --python 3.10 /tmp/tenv')
sh(f'uv pip install -q --python {PY} torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121')
sh(f'uv pip install -q --python {PY} diffusers==0.30.3 transformers==4.44.2 accelerate==0.33.0 peft==0.12.0 huggingface_hub==0.25.2 "numpy<2" ftfy tensorboard Jinja2 pillow safetensors datasets==2.21.0')
sh(f'git clone -q --depth 1 --branch {TAG} https://github.com/huggingface/diffusers.git {DF}')


In [ ]:
# LoRA data: the 81 real unripe training crops only. Inpainting sources: the 217 ripe crops and their masks. No validation or test image is read.
recs=pool_manifest['records']; assert all(r['split']=='train' for r in recs)
DS=WORK/'unripe'; DS.mkdir()
for r in recs:
    if r['domain']=='A': shutil.copy(POOL512/r['image'],DS/f"{r['id']}.png")
assert len(list(DS.iterdir()))==81
RIPE=sorted((r for r in recs if r['domain']=='B'),key=lambda r:r['id']); assert len(RIPE)==217
PROMPT='a photo of sks unripe melon'


In [ ]:
# LoRA settings fixed before any image was generated: diffusers defaults plus 2000 steps, batch 1, 512 px, fp16.
import threading, time
SEEDS=[11,12,13]; STEPS=2000
ENV=dict(os.environ,HF_HUB_DISABLE_TELEMETRY='1')
BASE='stable-diffusion-v1-5/stable-diffusion-v1-5'
def lora(seed,gpu):
    t=time.time(); log=open(WORK/f'lora_s{seed}.log','w')
    r=subprocess.run([PY,str(DF/'examples/dreambooth/train_dreambooth_lora.py'),'--pretrained_model_name_or_path',BASE,'--instance_data_dir',str(DS),
      '--instance_prompt',PROMPT,'--output_dir',str(WORK/f'lora_s{seed}'),'--resolution','512','--train_batch_size','1','--gradient_accumulation_steps','1',
      '--learning_rate','1e-4','--lr_scheduler','constant','--lr_warmup_steps','0','--max_train_steps',str(STEPS),'--checkpointing_steps',str(STEPS*2),
      '--mixed_precision','fp16','--seed',str(seed)],env=dict(ENV,CUDA_VISIBLE_DEVICES=str(gpu)),stdout=log,stderr=subprocess.STDOUT)
    print('lora',seed,'rc',r.returncode,'minutes',round((time.time()-t)/60,1),flush=True)
    if r.returncode: print(open(WORK/f'lora_s{seed}.log',errors='ignore').read()[-2500:],flush=True)
    return r.returncode
plan={0:[11,13],1:[12]}; results={}
def worker(g):
    for sd in plan[g]: results[sd]=lora(sd,g)
th=[threading.Thread(target=worker,args=(g,)) for g in plan]; [t.start() for t in th]; [t.join() for t in th]
if any(results.values()): raise SystemExit('LoRA training failed')
(WORK/'infer.py').write_text("""
import json, sys, torch
from pathlib import Path
from PIL import Image
from diffusers import StableDiffusionInpaintPipeline
lora, items, dst, seed, prompt = sys.argv[1], json.load(open(sys.argv[2])), Path(sys.argv[3]), int(sys.argv[4]), sys.argv[5]
p = StableDiffusionInpaintPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-inpainting', torch_dtype=torch.float16, safety_checker=None).to('cuda')
p.load_lora_weights(lora); p.set_progress_bar_config(disable=True); dst.mkdir(parents=True, exist_ok=True)
for i, x in enumerate(items):
    im = Image.open(x['image']).convert('RGB'); mk = Image.open(x['mask']).convert('L').point(lambda v: 255 if v > 127 else 0)
    g = torch.Generator('cuda').manual_seed(seed*10000+i)
    out = p(prompt, image=im, mask_image=mk, height=512, width=512, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0]
    out.save(dst/f"{x['id']}.png")
""")
(WORK/'items.json').write_text(json.dumps([dict(id=r['id'],image=str(POOL512/r['image']),mask=str(POOL512/r['mask'])) for r in RIPE]))
def infer(seed,gpu):
    t=time.time(); subprocess.run([PY,str(WORK/'infer.py'),str(WORK/f'lora_s{seed}'),str(WORK/'items.json'),str(WORK/f'raw_s{seed}'),str(seed),PROMPT],env=dict(ENV,CUDA_VISIBLE_DEVICES=str(gpu)),check=True)
    print('infer',seed,'minutes',round((time.time()-t)/60,1),flush=True)
def worker2(g):
    for sd in plan[g]: infer(sd,g)
th=[threading.Thread(target=worker2,args=(g,)) for g in plan]; [t.start() for t in th]; [t.join() for t in th]


In [ ]:
# Paste through the same masks and code as the other arms, then keep only what the local metrics and kernel 33 need.
from PIL import Image
import csv
from melon_cyclegan import config as cfg
gbc=s.tool('generate_balance_candidates')
rows={r['id']:r for r in csv.DictReader(open(DATASET/'manifest.csv'))}
byid={r['id']:r for r in pool_manifest['records']}
OUT=WORK/'result'; OUT.mkdir(); (OUT/'lora').mkdir()
for sd in SEEDS:
    raw=WORK/f'raw_s{sd}'; d=OUT/f'candidates/INPAINT_s{sd}'
    for sub in ('full/images','full/labels','images'): (d/sub).mkdir(parents=True)
    recs_=[]
    for f in sorted(raw.glob('*.png')):
        crop=byid[f.stem]; row=rows[f.stem]
        with Image.open(DATASET/row['image']) as h: full=h.convert('RGB')
        syn,clip=gbc.paste(full,Image.open(f).convert('RGB'),Image.open(POOL512/crop['mask']).convert('L'),crop['crop_window'])
        cid=f'balance__{f.stem}'
        syn.save(d/'full/images'/f'{cid}.jpg',quality=95)
        (d/'full/labels'/f'{cid}.txt').write_text(gbc.flipped_label((DATASET/row['label']).read_text(),cfg.UNRIPE))
        l,t,sz=crop['crop_window']; syn.crop((l,t,l+sz,t+sz)).resize((512,512),Image.Resampling.LANCZOS).save(d/'images'/f'{cid}.png')
        recs_.append(dict(candidate_id=cid,parent_id=f.stem,full_image=f'full/images/{cid}.jpg',full_label=f'full/labels/{cid}.txt',image=f'images/{cid}.png'))
    if len(recs_)!=217: raise SystemExit(f'seed {sd}: {len(recs_)} candidates')
    (d/'records.json').write_text(json.dumps(recs_,indent=1))
    shutil.copy(WORK/f'lora_s{sd}/pytorch_lora_weights.safetensors',OUT/f'lora/lora_s{sd}.safetensors'); shutil.copy(WORK/f'lora_s{sd}.log',OUT/f'lora/train_s{sd}.log')
(OUT/'summary.json').write_text(json.dumps(dict(baseline='SD inpainting with LoRA',diffusers='0.30.3',lora_base='stable-diffusion-v1-5/stable-diffusion-v1-5',inpaint='stable-diffusion-v1-5/stable-diffusion-inpainting',steps=STEPS,batch=1,size=512,prompt=PROMPT,guidance=7.5,sampling_steps=30,seeds=SEEDS,test_used=False),indent=1))
for name in ('unripe','raw_s11','raw_s12','raw_s13','lora_s11','lora_s12','lora_s13','study','code'): shutil.rmtree(WORK/name,ignore_errors=True)
print('done',sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file())//2**20,'MiB')
